In [ ]:
"""
Social Text Analytics — Climate Change Reddit Comments
Covers: Keyword frequency, TF-IDF, VADER sentiment, Controversiality analysis
"""

import os, re, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from collections import Counter

from sklearn.feature_extraction.text import TfidfVectorizer
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

warnings.filterwarnings("ignore")


In [ ]:

# ── Config ─────────────────────────────────────────────────────────────────────
DATA_PATH = "cleaned_climate_comments_2025_05_to_2026_04.xls"
OUT_DIR   = "figures"
os.makedirs(OUT_DIR, exist_ok=True)

STOPWORDS = {
    "the","a","an","is","it","in","on","at","to","for","of","and","or","but",
    "be","are","was","were","been","not","that","this","with","from","they",
    "have","has","had","he","she","we","you","i","my","your","our","their",
    "its","by","do","did","does","so","if","as","up","out","no","can","will",
    "just","like","get","go","got","also","more","than","then","what","how",
    "all","one","even","would","could","should","about","some","there","when",
    "which","who","his","her","him","them","me","us","s","t","re","ve","ll",
    "amp","gt","lt","www","http","https","reddit","subreddit","comment","post",
    "really","very","much","well","good","bad","think","know","people","don",
    "doesn","isn","won","said","say","says","still","way","many","make","made",
    "need","want","lot","things","thing","time","years","year","2024","2025","2026",
}

FOCUS_SUBREDDITS = [
    "climateskeptics","climatechange","climate","ClimateShitposting",
    "ClimateOffensive","ClimateActionPlan","GlobalClimateChange",
    "ClimateMemes","CitizensClimateLobby","Climate_Nuremberg","ClimateCO",
    "conspiracy","science","politics","worldnews","environment",
]


In [ ]:

def clean_text(text):
    if not isinstance(text, str): return ""
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    return text.lower().strip()

def tokenize(text):
    return [w for w in clean_text(text).split() if w not in STOPWORDS and len(w) > 2]

def save(fig, name):
    path = os.path.join(OUT_DIR, name)
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved → {path}")


In [ ]:

# ── Load ───────────────────────────────────────────────────────────────────────
print("Loading data …")
df = pd.read_csv(DATA_PATH, usecols=[
    "comment_id","score","self_text","subreddit","controversiality",
    "comment_month","comment_word_count","post_title"
])
df["self_text"] = df["self_text"].fillna("")
df["controversial"] = df["controversiality"].astype(int)
df["comment_month"] = pd.to_datetime(df["comment_month"], errors="coerce")
print(f"  {len(df):,} rows, {df['subreddit'].nunique()} subreddits")


In [ ]:

# Sample for tokenisation-heavy steps
SAMPLE = 80_000
sample_df = df.sample(SAMPLE, random_state=42).copy()
print(f"  Working sample for sentiment/keywords: {SAMPLE:,}")

# Tokenize sample
print("  Tokenizing …")
sample_df["tokens"] = sample_df["self_text"].apply(tokenize)

# Also tokenize controversial/non-controversial subsets from full df
# (do only 30K per group to stay fast)
cont1 = df[df["controversial"] == 1].sample(min(15000, df["controversial"].sum()), random_state=1)
cont0 = df[df["controversial"] == 0].sample(15000, random_state=2)
cont1["tokens"] = cont1["self_text"].apply(tokenize)
cont0["tokens"] = cont0["self_text"].apply(tokenize)


In [ ]:

# ── 1. Global keyword frequency ────────────────────────────────────────────────
print("\n[1] Keyword frequency …")
all_tokens = [t for ts in sample_df["tokens"] for t in ts]
freq       = Counter(all_tokens)
top_n      = freq.most_common(30)

words, counts = zip(*top_n)
fig, ax = plt.subplots(figsize=(12, 7))
ax.barh(words[::-1], counts[::-1], color=sns.color_palette("Blues_d", 30))
ax.set_xlabel("Frequency", fontsize=11)
ax.set_title("Top 30 Keywords — All Subreddits (80K sample)", fontsize=13, fontweight="bold")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"{x:,.0f}"))
plt.tight_layout()
save(fig, "01_keyword_frequency_global.png")

# Per-subreddit keyword heatmap
print("  Per-subreddit keyword heatmap …")
sub_df = sample_df[sample_df["subreddit"].isin(FOCUS_SUBREDDITS)]
sub_freqs = {}
for sub, grp in sub_df.groupby("subreddit"):
    toks = [t for ts in grp["tokens"] for t in ts]
    sub_freqs[sub] = Counter(toks)

sub_all = Counter()
for c in sub_freqs.values(): sub_all.update(c)
top_sub_words = [w for w, _ in sub_all.most_common(20)]

heat_data = pd.DataFrame(
    {sub: [sub_freqs[sub].get(w, 0) for w in top_sub_words] for sub in sub_freqs},
    index=top_sub_words,
).T
heat_norm = heat_data.div(heat_data.sum(axis=1).replace(0,1), axis=0) * 1000

fig, ax = plt.subplots(figsize=(16, 8))
sns.heatmap(heat_norm, cmap="YlOrRd", linewidths=0.3, ax=ax,
            cbar_kws={"label": "Relative freq (‰)"})
ax.set_title("Keyword Relative Frequency per Subreddit (top 20 terms)", fontsize=13, fontweight="bold")
ax.set_xlabel("Keyword"); ax.set_ylabel("Subreddit")
plt.xticks(rotation=40, ha="right")
plt.tight_layout()
save(fig, "02_keyword_heatmap_subreddits.png")


In [ ]:

# ── 2. TF-IDF per subreddit ────────────────────────────────────────────────────
print("\n[2] TF-IDF per subreddit …")
sub_docs = (
    sub_df.groupby("subreddit")["self_text"]
    .apply(lambda x: " ".join(x.dropna().apply(clean_text)))
    .reset_index()
)
sub_docs.columns = ["subreddit", "text_clean"]

vectorizer = TfidfVectorizer(
    max_features=5000, stop_words=list(STOPWORDS),
    ngram_range=(1, 2), min_df=2,
)
tfidf_matrix = vectorizer.fit_transform(sub_docs["text_clean"])
feature_names = vectorizer.get_feature_names_out()

tfidf_records = []
for i, sub in enumerate(sub_docs["subreddit"]):
    row     = tfidf_matrix[i].toarray().flatten()
    top_idx = row.argsort()[::-1][:10]
    for rank, idx in enumerate(top_idx):
        tfidf_records.append({"subreddit": sub, "rank": rank+1,
                               "term": feature_names[idx], "tfidf": round(float(row[idx]),5)})

tfidf_df = pd.DataFrame(tfidf_records)
tfidf_df.to_csv(os.path.join(OUT_DIR, "03_tfidf_top10_per_subreddit.csv"), index=False)
print("  Saved → figures/03_tfidf_top10_per_subreddit.csv")

plot_subs = [s for s in FOCUS_SUBREDDITS if s in sub_docs["subreddit"].values][:12]
n_cols = 4
n_rows = int(np.ceil(len(plot_subs) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, n_rows * 3.2))
axes = axes.flatten()
palette = sns.color_palette("tab20", 10)

for ax, sub in zip(axes, plot_subs):
    sub_data = tfidf_df[tfidf_df["subreddit"] == sub].head(8)
    if sub_data.empty: ax.set_visible(False); continue
    ax.barh(sub_data["term"][::-1].values, sub_data["tfidf"][::-1].values, color=palette)
    ax.set_title(f"r/{sub}", fontsize=9, fontweight="bold")
    ax.tick_params(axis="y", labelsize=7); ax.tick_params(axis="x", labelsize=7)
for ax in axes[len(plot_subs):]: ax.set_visible(False)
fig.suptitle("Top TF-IDF Terms per Subreddit (unigrams + bigrams)", fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
save(fig, "03_tfidf_facets.png")


In [ ]:

# ── 3. VADER Sentiment ─────────────────────────────────────────────────────────
print("\n[3] VADER sentiment …")
analyzer = SentimentIntensityAnalyzer()
print(f"  Scoring {SAMPLE:,} comments …")

scores = sample_df["self_text"].apply(
    lambda t: analyzer.polarity_scores(t if isinstance(t, str) else "")
)
sample_df["vader_neg"]      = scores.apply(lambda s: s["neg"])
sample_df["vader_pos"]      = scores.apply(lambda s: s["pos"])
sample_df["vader_compound"] = scores.apply(lambda s: s["compound"])
sample_df["sentiment_label"] = pd.cut(
    sample_df["vader_compound"],
    bins=[-1.01, -0.05, 0.05, 1.01],
    labels=["Negative", "Neutral", "Positive"],
)

# 3a. Global distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(sample_df["vader_compound"], bins=60, color="#4C72B0", edgecolor="white", linewidth=0.3)
axes[0].axvline(0, color="red", linestyle="--", linewidth=1.2, label="Neutral boundary")
axes[0].set_xlabel("VADER Compound Score", fontsize=11); axes[0].set_ylabel("Count", fontsize=11)
axes[0].set_title("Distribution of VADER Compound Scores", fontsize=12, fontweight="bold")
axes[0].legend()

label_counts = sample_df["sentiment_label"].value_counts()
axes[1].pie(label_counts, labels=label_counts.index, autopct="%1.1f%%",
            colors=["#d62728","#aec7e8","#2ca02c"], startangle=140, textprops={"fontsize":11})
axes[1].set_title("Sentiment Label Distribution", fontsize=12, fontweight="bold")
plt.tight_layout()
save(fig, "04_vader_global_distribution.png")

# 3b. Mean sentiment by subreddit
print("  Sentiment by subreddit …")
sub_sent = (
    sample_df[sample_df["subreddit"].isin(FOCUS_SUBREDDITS)]
    .groupby("subreddit")["vader_compound"]
    .agg(["mean","median","std","count"])
    .reset_index()
    .rename(columns={"mean":"mean_compound","median":"median_compound",
                     "std":"std_compound","count":"n_comments"})
    .sort_values("mean_compound")
)
sub_sent.to_csv(os.path.join(OUT_DIR, "04_sentiment_by_subreddit.csv"), index=False)

fig, ax = plt.subplots(figsize=(11, 7))
colors = ["#d62728" if v < 0 else "#2ca02c" for v in sub_sent["mean_compound"]]
ax.barh(sub_sent["subreddit"], sub_sent["mean_compound"], color=colors)
ax.axvline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_xlabel("Mean VADER Compound Score", fontsize=11)
ax.set_title("Mean Sentiment by Subreddit", fontsize=13, fontweight="bold")
ax.set_xlim(-0.25, 0.25)
plt.tight_layout()
save(fig, "04_sentiment_by_subreddit.png")

# 3c. Sentiment over time
print("  Sentiment over time …")
monthly = (
    sample_df.dropna(subset=["comment_month"])
    .groupby("comment_month")["vader_compound"]
    .agg(["mean","median"]).reset_index()
)
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(monthly["comment_month"], monthly["mean"],   marker="o", label="Mean",   color="#4C72B0")
ax.plot(monthly["comment_month"], monthly["median"], marker="s", linestyle="--", label="Median", color="#DD8452")
ax.axhline(0, color="grey", linewidth=0.7, linestyle=":")
ax.set_xlabel("Month", fontsize=11); ax.set_ylabel("VADER Compound Score", fontsize=11)
ax.set_title("Sentiment Trend Over Time", fontsize=13, fontweight="bold"); ax.legend()
plt.xticks(rotation=30); plt.tight_layout()
save(fig, "04_sentiment_trend_monthly.png")

# 3d. Stacked sentiment composition per subreddit
sub_label = (
    sample_df[sample_df["subreddit"].isin(FOCUS_SUBREDDITS)]
    .groupby(["subreddit","sentiment_label"]).size().unstack(fill_value=0)
)
sub_label_pct = sub_label.div(sub_label.sum(axis=1), axis=0) * 100
sub_label_pct = sub_label_pct.sort_values("Negative", ascending=False)
fig, ax = plt.subplots(figsize=(12, 7))
sub_label_pct.plot(kind="barh", stacked=True, ax=ax, width=0.7,
                   color={"Negative":"#d62728","Neutral":"#aec7e8","Positive":"#2ca02c"})
ax.set_xlabel("% of comments", fontsize=11)
ax.set_title("Sentiment Composition per Subreddit", fontsize=13, fontweight="bold")
ax.legend(title="Sentiment", loc="lower right"); plt.tight_layout()
save(fig, "04_sentiment_stacked_subreddits.png")


In [ ]:

# ── 4. Controversiality ────────────────────────────────────────────────────────
print("\n[4] Controversiality analysis …")

# 4a. Rate by subreddit (full df)
cont_rate = (
    df[df["subreddit"].isin(FOCUS_SUBREDDITS)]
    .groupby("subreddit")["controversial"]
    .agg(["mean","sum","count"]).reset_index()
    .rename(columns={"mean":"controversy_rate","sum":"n_controversial","count":"n_total"})
    .sort_values("controversy_rate", ascending=False)
)
cont_rate.to_csv(os.path.join(OUT_DIR, "05_controversiality_by_subreddit.csv"), index=False)

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(cont_rate["subreddit"][::-1], cont_rate["controversy_rate"][::-1] * 100,
        color=sns.color_palette("Reds_r", len(cont_rate)))
ax.set_xlabel("Controversy Rate (%)", fontsize=11)
ax.set_title("Controversiality Rate by Subreddit", fontsize=13, fontweight="bold")
ax.xaxis.set_major_formatter(mticker.PercentFormatter())
plt.tight_layout()
save(fig, "05_controversiality_by_subreddit.png")

# 4b. Sentiment vs controversiality (use sample_df)
cont_sent = (
    sample_df[sample_df["subreddit"].isin(FOCUS_SUBREDDITS)]
    .groupby(["subreddit","controversial"])["vader_compound"]
    .mean().unstack().reset_index()
)
cont_sent.columns = ["subreddit","not_controversial","controversial"]
cont_sent = cont_sent.dropna().sort_values("not_controversial")

x = np.arange(len(cont_sent)); w = 0.35
fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(x - w/2, cont_sent["not_controversial"], w, label="Not controversial", color="#4C72B0")
ax.bar(x + w/2, cont_sent["controversial"],     w, label="Controversial",     color="#d62728")
ax.set_xticks(x)
ax.set_xticklabels(cont_sent["subreddit"], rotation=40, ha="right", fontsize=8)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_ylabel("Mean VADER Compound Score", fontsize=11)
ax.set_title("Sentiment: Controversial vs. Non-Controversial Comments", fontsize=12, fontweight="bold")
ax.legend(); plt.tight_layout()
save(fig, "05_sentiment_vs_controversiality.png")

# 4c. Controversiality over time (full df)
cont_monthly = (
    df.dropna(subset=["comment_month"])
    .groupby("comment_month")["controversial"]
    .agg(["mean","sum","count"]).reset_index()
)
fig, ax1 = plt.subplots(figsize=(12, 5))
ax1.bar(cont_monthly["comment_month"], cont_monthly["sum"], width=20,
        color="#d62728", alpha=0.5, label="# Controversial comments")
ax1.set_ylabel("Count", color="#d62728", fontsize=11)
ax1.tick_params(axis="y", labelcolor="#d62728")
ax2 = ax1.twinx()
ax2.plot(cont_monthly["comment_month"], cont_monthly["mean"] * 100,
         color="#333333", marker="o", linewidth=1.5, label="Rate (%)")
ax2.set_ylabel("Controversy Rate (%)", fontsize=11)
ax1.set_title("Controversiality Over Time", fontsize=13, fontweight="bold")
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")
plt.xticks(rotation=30); plt.tight_layout()
save(fig, "05_controversiality_trend.png")

# 4d. Top keywords: controversial vs non-controversial
for label_val, grp_df, tag in [
    (1, cont1, "controversial"),
    (0, cont0, "non_controversial"),
]:
    toks = [t for ts in grp_df["tokens"] for t in ts]
    top20 = Counter(toks).most_common(20)
    words_l, counts_l = zip(*top20)
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.barh(words_l[::-1], counts_l[::-1], color="#d62728" if label_val==1 else "#4C72B0")
    ax.set_title(f"Top Keywords — {'Controversial' if label_val==1 else 'Non-Controversial'} Comments",
                 fontsize=12, fontweight="bold")
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"{x:,.0f}"))
    plt.tight_layout()
    save(fig, f"06_keywords_{tag}.png")

# ── 5. Summary ─────────────────────────────────────────────────────────────────
print("\n[5] Writing summary CSV …")
summary = (
    df[df["subreddit"].isin(FOCUS_SUBREDDITS)]
    .groupby("subreddit")
    .agg(
        n_comments       = ("comment_id",       "count"),
        mean_score       = ("score",            "mean"),
        median_score     = ("score",            "median"),
        n_controversial  = ("controversial",    "sum"),
        controversy_rate = ("controversial",    "mean"),
        mean_word_count  = ("comment_word_count","mean"),
    )
    .reset_index().round(4)
)
summary.to_csv(os.path.join(OUT_DIR, "00_subreddit_summary.csv"), index=False)
print("  Saved → figures/00_subreddit_summary.csv")
print("\n✓ All done.")
